In [16]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_1203203_unopt_rPBE_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [17]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [18]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [19]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [20]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [21]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [22]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[ -32.40339657  199.69003194   58.02439206]
 [ 174.78024519  129.93785018  -82.66612361]
 [  17.7304613   -42.34977871 -146.76410009]]

17O2 sigma:
 [[ -32.40339657 -199.69003194  -58.02439206]
 [-174.78024519  129.93785018  -82.66612361]
 [ -17.7304613   -42.34977871 -146.76410009]]

17O3 sigma:
 [[ -32.40339657  199.69003194  -58.02439206]
 [ 174.78024519  129.93785018   82.66612361]
 [ -17.7304613    42.34977871 -146.76410009]]

17O4 sigma:
 [[ -32.40339657 -199.69003194   58.02439206]
 [-174.78024519  129.93785018   82.66612361]
 [  17.7304613    42.34977871 -146.76410009]]

17O5 sigma:
 [[-46.97895011 165.63038908 -76.19400418]
 [169.06635011  91.28617237  27.30033548]
 [  8.47562209 -22.94989151 -40.14639128]]

17O6 sigma:
 [[ -46.97895011 -165.63038908   76.19400418]
 [-169.06635011   91.28617237   27.30033548]
 [  -8.47562209  -22.94989151  -40.14639128]]

17O7 sigma:
 [[-46.97895011 165.63038908  76.19400418]
 [169.06635011  91.28617237 -27.30033548]
 [ -8.475622

In [23]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 8.20810849825939

17O2 sigma:
 8.208108498259378

17O3 sigma:
 8.208108498259419

17O4 sigma:
 8.208108498259415

17O5 sigma:
 6.67995095613694

17O6 sigma:
 6.679950956136945

17O7 sigma:
 6.679950956136849

17O8 sigma:
 6.679950956136861



In [24]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-4.262  0.919 -1.22 ]
 [ 0.919 -3.625  1.654]
 [-1.22   1.654  7.887]]

CS Tensor:
 [[ -32.403  199.69    58.024]
 [ 174.78   129.938  -82.666]
 [  17.73   -42.35  -146.764]]

CS isotropic Tensor:
 [[-16.41   0.     0.  ]
 [  0.   -16.41   0.  ]
 [  0.     0.   -16.41]]

CS symmetric Tensor:
 [[ -32.403  187.235   37.877]
 [ 187.235  129.938  -62.508]
 [  37.877  -62.508 -146.764]]

CS antisymmetric Tensor:
 [[  0.     12.455  20.147]
 [-12.455   0.    -20.158]
 [-20.147  20.158   0.   ]]


In [25]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 8.21452328 -5.21151875 -3.00300453] 

 Unsorted Eigenvectors:
 [[-0.08691371 -0.78011947  0.61956405]
 [ 0.13117526  0.60754075  0.78338196]
 [ 0.98754193 -0.14935811 -0.04952876]] 

Sorted Eigenvalues: 
 [-3.00300453 -5.21151875  8.21452328] 

Sorted Eigenvectors: 
 [[ 0.61956405 -0.78011947 -0.08691371]
 [ 0.78338196  0.60754075  0.13117526]
 [-0.04952876 -0.14935811  0.98754193]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 255.36947167  -86.46822652 -218.13089162] 

 Unsorted Eigenvectors:
 [[-0.53617986 -0.61378428 -0.57946529]
 [-0.84029349  0.32296639  0.43543031]
 [ 0.08011246 -0.72038987  0.68892701]] 

Sorted Eigenvalues: 
 [ -86.46822652 -218.13089162  255.36947167] 

Sorted Eigenvectors: 
 [[-0.61378428 -0.57946529 -0.53617986]
 [ 0.32296639  0.43543031 -0.84029349]
 [-0.72038987  0.68892701  0.08011246]] 



In [26]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.003004528602359 -5.211518753738244 8.214523282340531
CSA Tensor Components δyy, δxx, δzz: 
 -86.4682265220384 -218.13089162071054 255.3694716712036


In [27]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   8.21452  |
+--------------+------------+
| etaq         |   0.268855 |
+--------------+------------+
| iso_cs (ppm) | -16.4099   |
+--------------+------------+
| csa (ppm)    | 271.779    |
+--------------+------------+
| etas         |   0.484447 |
+--------------+------------+


In [28]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [29]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.78011947  0.61956405 -0.08691371]
 [ 0.60754075  0.78338196  0.13117526]
 [-0.14935811 -0.04952876  0.98754193]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
18.346077856650886 9.053467957248092 56.47246280608675 

Direction cosine csa: 

[[-0.57946529 -0.61378428 -0.53617986]
 [ 0.43543031  0.32296639 -0.84029349]
 [ 0.68892701 -0.72038987  0.08011246]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-46.2789101593044 85.4049697676858 -57.45860608469686 



In [30]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -38.00844481132807 chi: 89.11244855787614 xi: -84.01856908760642 

